# Lab 02 — PyTorch → ONNX export (guided exercises)

**Prerequisites:** finished `chapter_02_pytorch_to_onnx.ipynb` and `chapter_03_onnxruntime_inference.ipynb`.

**Goal:** practice the export workflow by repeating it on two different architectures, comparing file sizes and inference latency, and intentionally triggering one common export error so you recognize it next time.

Each exercise has a TODO cell — fill it in, run it, and verify against the success criteria.

Run from the repo root.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import onnx
import onnxruntime as ort
import torch
from torchvision import models

OUT = Path('../experiments/exported_models')
OUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — Export ResNet-18

**Task:** export `torchvision.models.resnet18` with default weights to `experiments/exported_models/resnet18.onnx`, opset 17, single file (`dynamo=False`).

**Success criteria:**
- File exists.
- `onnx.checker.check_model` passes.
- File size is roughly 45 MB ± 5 MB.

In [ ]:
# TODO Exercise 1: export resnet18 here.
# Hint: see how Chapter 2 notebook did it for MobileNetV3-Small.

# ----- Verification -----
resnet_path = OUT / 'resnet18.onnx'
if resnet_path.exists():
    size = resnet_path.stat().st_size / 1024 / 1024
    onnx.checker.check_model(onnx.load(resnet_path.as_posix()))
    print(f'OK: {resnet_path}  size={size:.2f} MB')
    assert 35 < size < 55, f'Expected ~45 MB, got {size:.2f} MB'
else:
    print('NOT YET: export resnet18 first')

## Exercise 2 — Compare predictions between PyTorch and ONNX

**Task:** for the ResNet-18 you just exported, sample 5 random inputs of shape `(1, 3, 224, 224)`, run them through both PyTorch and ONNX Runtime, and report the max abs diff and same-argmax rate.

**Success criteria:** max abs diff < 1e-3 across all 5 inputs, same_argmax_rate = 5/5.

In [ ]:
# TODO Exercise 2: fill in the loop below.
weights = models.ResNet18_Weights.IMAGENET1K_V1
torch_model = models.resnet18(weights=weights).eval()
sess = ort.InferenceSession(resnet_path.as_posix(), providers=['CPUExecutionProvider'])
in_name = sess.get_inputs()[0].name

max_diffs = []
same_argmax = 0
for _ in range(5):
    # TODO: create a random input, run both, append max abs diff to max_diffs, increment same_argmax if matching
    pass

print('max abs diffs:', max_diffs)
print('same argmax rate:', f'{same_argmax}/5')
# assert max(max_diffs) < 1e-3, 'Diff too large — investigate'
# assert same_argmax == 5, 'argmax disagreement — investigate'

## Exercise 3 — Trigger an opset error on purpose

**Task:** try to export the same ResNet-18 with `opset_version=7` (very old). Observe the error message, then catch it and print just the first line.

**Why:** when you target old runtimes (TensorRT on JetPack 5, OpenVINO 2022), this exact failure happens. Knowing the message helps you fix it fast.

In [ ]:
# TODO Exercise 3: try the bad export inside try/except. Print only the first line of the error.
try:
    pass  # TODO
except Exception as e:
    msg = str(e).splitlines()[0]
    print('CAUGHT:', msg)

## Exercise 4 — Compare file sizes

Fill in the table below and answer:

1. Which model has the smallest `.onnx`?
2. Is the model size proportional to the parameter count? (Hint: parameter dtype matters — FP32 = 4 bytes/param.)
3. Which model would you ship to a Raspberry Pi 4 with 1 GB RAM?

| Model | params (M) | `.onnx` size (MB) | params × 4 / 1e6 |
|---|---|---|---|
| mobilenet_v3_small | ? | ? | ? |
| resnet18 | ? | ? | ? |
| resnet50 (export it) | ? | ? | ? |

In [ ]:
# TODO Exercise 4: export resnet50 if you haven't yet, then fill in the table.
for name, builder, w in [
    ('mobilenet_v3_small', models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.IMAGENET1K_V1),
    ('resnet18',           models.resnet18,           models.ResNet18_Weights.IMAGENET1K_V1),
    ('resnet50',           models.resnet50,           models.ResNet50_Weights.IMAGENET1K_V2),
]:
    m = builder(weights=w)
    params = sum(p.numel() for p in m.parameters())
    onnx_path = OUT / f'{name}.onnx'
    size_mb = onnx_path.stat().st_size / 1024 / 1024 if onnx_path.exists() else float('nan')
    print(f'{name:<25s} params={params/1e6:7.2f} M  onnx={size_mb:7.2f} MB  expected≈{params*4/1024/1024:.2f} MB')

## Take-aways

- Every model you export should land in `experiments/exported_models/`.
- Validate every export numerically; don't trust file existence alone.
- Watch for opset mismatches — they are the first thing to check when an old runtime rejects your file.
- File size ≈ `params × 4` bytes for FP32. Halve for FP16. Roughly quarter for INT8 (Chapter 8).